In [1]:
import glob
import os
import shutil

import pandas as pd

# Package data for ArcGIS Desktop

I have created "template" documents using ArcMap and ArcScene that display the NDVI and elevation datasets in a standardised, interactive way. This notebook copies the templates and updates them with the correct data for each mission.

In [2]:
# Read list of datasets from Mathias
ds_list_xlsx = r"../data/mathias_msi_missions.xlsx"
ds_df = pd.read_excel(ds_list_xlsx)

# Only process the ones without data issues
ds_df = ds_df.query("comment == 'OK'").reset_index(drop=True)

ds_df

,flight,minio_folder,mosaic_name,dsm_path,comment
0,H20,niva-tidy/2023/niva_202308301136_halden_h20_ms...,niva-sabicas_halden-h20_202308301136_msi_80m,niva-tidy/2023/niva_202308301149_halden_h20_rg...,OK
1,H21,niva-tidy/2023/niva_202308301118_halden_h21_ms...,niva-sabicas_halden-h21_202308301118_msi_80m,niva-tidy/2023/niva_202308301109_halden_h21_rg...,OK
2,H22,niva-tidy/2023/niva_202308300752_halden_h22_ms...,niva-sabicas_halden-h22_202308300752_msi_80m,niva-tidy/2023/niva_202308300741_halden_h22_rg...,OK
3,H23,niva-tidy/2023/niva_202308301044_halden_h23_ms...,niva-sabicas_halden-h23_202308301044_msi_80m,niva-tidy/2023/niva_202308301032_halden_h23_rg...,OK
4,H24,niva-tidy/2023/niva_202308291402_halden_h24_ms...,niva-sabicas_halden-h24_202308291402_msi_80m,niva-tidy/2023/niva_202308291354_halden_h24_rg...,OK
5,H25,niva-tidy/2023/niva_202308291309_halden_h25_ms...,niva-sabicas_halden-h25_202308291309_msi_80m,niva-tidy/2023/niva_202308291253_halden_h25_rg...,OK
6,H28,niva-tidy/2023/niva_202308291436_halden_h28_ms...,niva-sabicas_halden-h28_202308291436_msi_80m,niva-tidy/2023/niva_202308291429_halden_h28_rg...,OK
7,H29,niva-tidy/2023/niva_202308290959_halden_h29_ms...,niva-sabicas_halden-h29_202308290959_msi_80m,niva-tidy/2023/niva_202308290947_halden_h29_rg...,OK
8,H30,niva-tidy/2023/niva_202308291100_halden_h30_ms...,niva-sabicas_halden-h30_202308291100_msi_80m,niva-tidy/2023/niva_202308291046_halden_h30_rg...,OK
9,H31,niva-tidy/2023/niva_202308291138_halden_h31_ms...,niva-sabicas_halden-h31_202308291138_msi_80m,niva-tidy/2023/niva_202308291127_halden_h31_rg...,OK


In [3]:
base_fold = r"../gis"
src_mxd_path = os.path.join(base_fold, "doc_templates", "map_2d.mxd")
src_sxd_path = os.path.join(base_fold, "doc_templates", "map_3d.sxd")

for idx, row in ds_df.iterrows():
    folder = row["minio_folder"]
    name = row["mosaic_name"]
    site = row["flight"]
    rgb_dsm_path = row["dsm_path"]

    # Create output folder
    out_fold = os.path.join(base_fold, site.lower())
    if not os.path.exists(out_fold):
        os.makedirs(out_fold)

    # Get paths to data on MinIO
    rgb_fold = (
        f"/home/notebook/shared-seabee-ns9879k/{rgb_dsm_path.split('/orthophoto')[0]}"
    )
    search_path = os.path.join(rgb_fold, "orthophoto", "niva-sabicas*.tif")
    flist = glob.glob(search_path)
    assert len(flist) == 1
    rgb_tif_path = flist[0]
    ndvi_tif_path = (
        f"/home/notebook/shared-seabee-ns9879k/{folder}/ndvi/{name}_ndvi.tif"
    )
    dsm_tif_path = f"/home/notebook/shared-seabee-ns9879k/{folder}/rgb_dsm_resampled/{name}_dsm.tif"

    # Copy files
    dst_mxd_path = os.path.join(out_fold, "map_2d.mxd")
    shutil.copy(src_mxd_path, dst_mxd_path)

    dst_sxd_path = os.path.join(out_fold, "map_3d.sxd")
    shutil.copy(src_sxd_path, dst_sxd_path)

    dst_rgb_path = os.path.join(out_fold, "rgb.tif")
    shutil.copy(rgb_tif_path, dst_rgb_path)

    dst_ndvi_path = os.path.join(out_fold, "ndvi.tif")
    shutil.copy(ndvi_tif_path, dst_ndvi_path)

    dst_dsm_path = os.path.join(out_fold, "dsm.tif")
    shutil.copy(dsm_tif_path, dst_dsm_path)